# Proiect Final — Partea A: Analiză Generală

## Cuprins
1. [Formularea Problemei & Explorarea Datelor (EDA)](#1-formularea-problemei--eda)
2. [Preprocesare în Pipeline (Scalare, Codificare, Data Leakage)](#2-preprocesare-în-pipeline)
3. [Analiza Dezechilibrului (Justificarea Neacesității)](#3-dezechilibrul-claselor)
4. [Modele Propuși (Model Baseline, SVM, Ansamblu)](#4-modele-implementate)
5. [Design-ul Mulțimilor de Antrenament](#5-design-mulțimi-de-antrenament)
6. [Validare Încrucișată (Cross-Validation)](#6-cross-validation)
7. [Selecția Modelelor (Grid Search)](#7-model-selection)
8. [Metrici de Evaluare & Compararea Modelelor](#8-metrici-și-comparare)
9. [Interpretarea Rezultatelor și Concluzii](#9-interpretare-și-concluzii)


# 1. Formularea Problemei & Explorarea Datelor (EDA)

## <span style="color: #5bb04a;">1.1 Formularea Problemei din Contextul Datelor</span>
* **Obiectivul Setului de Date:** Datele monitorizează condițiile climatice dintr-o casă inteligentă cu consum redus de energie pe o perioadă de 4.5 luni, folosind senzori wireless care preiau datele la intervale de 10 minute. Datele climatice interne sunt combinate cu datele meteorologice de la cea mai apropiată stație meteo (Chievres, Belgia).
* **Variabila Țintă (Target):** `Appliances` — Exprimă consumul de energie al electrocasnicelor din locuință în watt pe oră (Wh).
* **Tipul Problemei:** Aceasta este o problemă de **Regresie Supervizată**. Scopul este aproximarea unei funcții matematice capabile să anticipeze consumul viitor de energie pe baza vectorului de caracteristici climatice.
* **Semnificație Practică:** Estimarea precisă a consumului energetic permite implementarea unor sisteme inteligente de management al clădirilor pentru reducerea cheltuielilor și optimizarea sarcinii electrice în rețele de tip Smart Grid.


## <span style="color: #5bb04a;">1.2 Incarcarea datasetului</span>

In [8]:
import os
import pandas as pd

RAW_DATA_PATH = "../data/raw/energydata_complete.csv"

if not os.path.exists(RAW_DATA_PATH):
    print("Downloading raw dataset from UCI Repository...")
    
    direct_csv_url = "https://archive.ics.uci.edu/static/public/374/appliances+energy+prediction.zip"
    
    raw_df = pd.read_csv(direct_csv_url, compression = "zip")
    
    raw_df.to_csv(RAW_DATA_PATH, index=False)
    print(f"Success! Raw dataset saved in: {RAW_DATA_PATH}")
else:
    print("Raw dataset file already exists locally !")

Raw dataset file already exists locally !


## <span style="color: #5bb04a;"> 1.3 Studierea datasetului</span>

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/energydata_complete.csv")

print("=== 1. DIMENSIONALITY ARCHITECTURE ===")
print(f"Total Instances (Rows): {df.shape[0]}")
print(f"Total Attributes (Columns): {df.shape[1]}\n")

print("=== 2. DATA TYPE INTEGRITY SCAN ===")
print(df.info())
print("\n")

print("=== 3. MISSING VALUES ===")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "No missing values were detected.")
print("\n")

print("=== 4. DUPLICATION ANALYSIS ===")
duplicate_count = df.duplicated().sum()
print(f"Duplicate Rows Detected: {duplicate_count}\n")

print("=== 5. RANGE BOUNDARY SUMMARY ===")
df.describe().T

=== 1. DIMENSIONALITY ARCHITECTURE ===
Total Instances (Rows): 19735
Total Attributes (Columns): 29

=== 2. DATA TYPE INTEGRITY SCAN ===
<class 'pandas.DataFrame'>
RangeIndex: 19735 entries, 0 to 19734
Data columns (total 29 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         19735 non-null  str    
 1   Appliances   19735 non-null  int64  
 2   lights       19735 non-null  int64  
 3   T1           19735 non-null  float64
 4   RH_1         19735 non-null  float64
 5   T2           19735 non-null  float64
 6   RH_2         19735 non-null  float64
 7   T3           19735 non-null  float64
 8   RH_3         19735 non-null  float64
 9   T4           19735 non-null  float64
 10  RH_4         19735 non-null  float64
 11  T5           19735 non-null  float64
 12  RH_5         19735 non-null  float64
 13  T6           19735 non-null  float64
 14  RH_6         19735 non-null  float64
 15  T7           19735 non-null  float64
 16  RH_7

,count,mean,std,min,25%,50%,75%,max
Appliances,19735.0,97.694958,102.524891,10.000000,50.000000,60.000000,100.000000,1080.000000
lights,19735.0,3.801875,7.935988,0.000000,0.000000,0.000000,0.000000,70.000000
T1,19735.0,21.686571,1.606066,16.790000,20.760000,21.600000,22.600000,26.260000
RH_1,19735.0,40.259739,3.979299,27.023333,37.333333,39.656667,43.066667,63.360000
T2,19735.0,20.341219,2.192974,16.100000,18.790000,20.000000,21.500000,29.856667
RH_2,19735.0,40.420420,4.069813,20.463333,37.900000,40.500000,43.260000,56.026667
T3,19735.0,22.267611,2.006111,17.200000,20.790000,22.100000,23.290000,29.236000
RH_3,19735.0,39.242500,3.254576,28.766667,36.900000,38.530000,41.760000,50.163333
T4,19735.0,20.855335,2.042884,15.100000,19.530000,20.666667,22.100000,26.200000
RH_4,19735.0,39.026904,4.341321,27.660000,35.530000,38.400000,42.156667,51.090000


## <span style="color: #4a7bb0;">Observatii si concluzii</span>

### 1. Structura Datelor
- <strong><span style="color: #d65d5d;">Constatare</span>:</strong> Setul de date conține 19.735 de înregistrări și 29 de coloane. Toate atributele sunt numerice (reale sau întregi), cu excepția coloanei `date`, care este înregistrată inițial sub formă de text (`str`).
- <strong><span style="color: #5bb04a;">Decizie</span>:</strong> Vom converti coloana `date` în formatul nativ `datetime64` pentru a putea extrage caracteristici temporale (ora din zi, ziua săptămânii, luna), elemente vitale pentru modelare.

### 2. Verificarea Calității Datelor (Valori Nule și Duplicate)
- <strong><span style="color: #d65d5d;">Constatare</span>:</strong> Nu a fost identificata nicio valoare lipsă (*missing values*) și niciun rând duplicat în întregul set de date.
- <strong><span style="color: #5bb04a;">Decizie</span>:</strong> Vom simplifica arhitectura fluxului de preprocesare (*pipeline*), eliminând necesitatea de imputare a datelor (ex: `SimpleImputer`), ceea ce reduce redundanța si complexitatea codului.

### 3. Curățarea Atributelor de Control (Zgomot Artificial)
- <strong><span style="color: #d65d5d;">Constatare</span>:</strong> Variabilele `rv1` și `rv2` sunt, conform documentației oficiale UCI, variabile pur aleatoare (*Random Variables*) introduse special pentru a testa reziliența modelelor la zgomot.
- <strong><span style="color: #5bb04a;">Decizie</span>:</strong> Vom elimina definitiv din matricea de caracteristici aceste doua atribute pentru a preveni fenomenul de *overfitting*.

### 4. Analiza Distribuției Variabilei Țintă (`Appliances`)
- <strong><span style="color: #d65d5d;">Constatare</span>:</strong> Analiza statistică de bază (`.describe()`) arată un consum mediu de aproximativ 97 Wh, insa cu un prag maxim ce urcă până la 1.080 Wh. Acest lucru indica o distribuție asimetrică la dreapta (*right-skewed*) posibil generata de vârfuri ocazionale de consum.
- <strong><span style="color: #5bb04a;">Decizie</span>:</strong> Deoarece modelele liniare pot fi afectate de valori extreme și asimetrii, alegem sa aplicam o transformare logaritmică a țintei pentru a stabiliza varianța erorilor.


# 2. Preprocesare în Pipeline

## 2.1 Placeholder (NU UITA SA COMPLETEZI)

În acest capitol, transformăm deciziile luate în faza de analiză anterioara într-un flux de lucru automatizat și robust.

### 1. Transformarea atributului `date`
* <strong><span style="color: #d65d5d;">Constatare</span>:</strong> Coloana `date` (format calendaristic complex) nu poate fi procesată matematic de algoritmi precum regresia liniară sau SVM. 
* <strong><span style="color: #5bb04a;">Decizie</span>:</strong> Folosim această coloană pentru a extrage două atribute numerice ciclice: `Hour` (Ora din zi, 0-23) și `DayOfWeek` (Ziua din săptămână, 0-6) și eliminăm (`drop`) coloana brută pentru a nu bloca compilarea modelelor.

In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df['date'] = pd.to_datetime(df['date'])

df = df.sort_values(by='date').reset_index(drop=True)

df['Hour'] = df['date'].dt.hour
df['DayOfWeek'] = df['date'].dt.day_of_week

X = df.drop(columns=['date', 'Appliances', 'rv1', 'rv2']) # Eliminarea coloanelor irelevante pentru modelare

y = np.log1p(df['Appliances']) # Transformarea logaritmică a valorilor target

### 2. Designul mulțimii de antrenament și preveinerea fenomenului de `Data Leakage` 
* <strong><span style="color: #d65d5d;">Constatare</span>:</strong> Deoarece lucrăm cu o serie temporală, amestecarea aleatorie a datelor (`Shuffle`) ar permite modelului să folosească informații din "viitor" pentru a prezice "trecutul". De asemenea, dacă calculăm media sau deviația standard (pentru scalare) pe întregul set de date înainte de split, setul de antrenament va afla indicii statistice din setul de test.
* <strong><span style="color: #5bb04a;">Decizie</span>:</strong> 
  1. Sortăm întregul set de date strict cronologic.
  2. Realizăm o separare fixă: primele 80% din înregistrări devin setul de antrenament (**Train**), iar ultimele 20% devin setul de test (**Test**).
  3. Vom antrena transformatoarele statistice (cum este `StandardScaler`) **exclusiv** pe datele de Train, aplicându-le ulterior pe cele de Test ca o simplă transformare.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Total rânduri în setul inițial:  {len(df)}")
print(f"Rânduri alocate pentru Train (80%): {X_train.shape[0]}")
print(f"Rânduri alocate pentru Test  (20%): {X_test.shape[0]}")
print(f"Număr final de atribute în X:    {X_train.shape[1]}")
print("\n=== Caracteristici rămase în matricea X ===")
print(list(X_train.columns))


Total rânduri în setul inițial:  19735
Rânduri alocate pentru Train (80%): 15788
Rânduri alocate pentru Test  (20%): 3947
Număr final de atribute în X:    27

=== Caracteristici rămase în matricea X ===
['lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'Hour', 'DayOfWeek']


### 3. Strategia de scalare și Pipeline: 
Construim un `ColumnTransformer` care aplică `StandardScaler` doar pe senzorii continui (temperaturi, umiditate, presiune), lăsând orele și zilele intacte (`passthrough`). Pentru a compara performanța, vom antrena modelele în două moduri: direct pe datele brute și prin Pipeline-ul de scalare.

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

sensor_attributes = [col for col in X_train.columns if col not in ['Hour', 'DayOfWeek']]

climate_preprocessor = ColumnTransformer(
    transformers=[
        ('scalare_standard', StandardScaler(), sensor_attributes)
    ],
    remainder='passthrough'
)

print(f"Număr de atribute ce vor fi scalate: {len(sensor_attributes)}")
print(f"Atribute pass-through:               ['Hour', 'DayOfWeek']")

Număr de atribute ce vor fi scalate: 25
Atribute pass-through:               ['Hour', 'DayOfWeek']
